# Trips & Users — Cancellation Rate Per Day

**Question:** Given `trips` (id, client_id, driver_id, city_id, status, request_at) and `user_info` (user_info_id, banned, role) tables, find the cancellation rate (cancelled trips / total trips) for each date between 2013-10-01 and 2013-10-03, excluding trips where either the client or driver is banned. Solve in both SQL and PySpark.

In [0]:
%sql
-- Step 1: Create the trips table (id, client_id, driver_id, city_id, status, request_at)
-- Step 2: Create the user_info table (user_info_id, banned, role)
-- Step 3: Insert 10 trip records (Oct 1-3, 2013) and 8 user records (clients + drivers, one banned)

Create table if not exists b_sql.b_practice.trips (id int, client_id int, driver_id int, city_id int, status varchar(50), request_at varchar(50));
Create table b_sql.b_practice.user_info (user_info_id int, banned varchar(50), role varchar(50));
Truncate table b_sql.b_practice.trips;
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('1', '1', '10', '1', 'completed', '2013-10-01');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('2', '2', '11', '1', 'cancelled_by_driver', '2013-10-01');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('3', '3', '12', '6', 'completed', '2013-10-01');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('4', '4', '13', '6', 'cancelled_by_client', '2013-10-01');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('5', '1', '10', '1', 'completed', '2013-10-02');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('6', '2', '11', '6', 'completed', '2013-10-02');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('7', '3', '12', '6', 'completed', '2013-10-02');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('8', '2', '12', '12', 'completed', '2013-10-03');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('9', '3', '10', '12', 'completed', '2013-10-03');
insert into b_sql.b_practice.trips (id, client_id, driver_id, city_id, status, request_at) values ('10', '4', '13', '12', 'cancelled_by_driver', '2013-10-03');
Truncate table b_sql.b_practice.user_info;
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('1', 'No', 'client');
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('2', 'Yes', 'client');
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('3', 'No', 'client');
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('4', 'No', 'client');
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('10', 'No', 'driver');
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('11', 'No', 'driver');
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('12', 'No', 'driver');
insert into b_sql.b_practice.user_info (user_info_id, banned, role) values ('13', 'No', 'driver');

In [0]:
%sql
-- Verify the trips data
select * from b_sql.b_practice.trips;

In [0]:
%sql
-- Verify the user_info data
select * from b_sql.b_practice.user_info

In [0]:
%sql
-- SQL Solution: Cancellation rate per day (excluding banned clients/drivers)
--
-- CTE: Join trips with user_info twice (for client and driver) on banned='No',
--       count total trips and sum cancelled (status != 'completed') per date,
--       filter to Oct 1-3, 2013.
-- Outer query: Compute cancellation percentage as round(cancelled*100/total, 2)
with cte as (select t.request_at, count(*) as total, sum(case when t.status != 'completed' then 1 else 0 end) as canceled from b_sql.b_practice.trips t join b_sql.b_practice.user_info u on t.client_id = u.user_info_id and  u.banned = 'No' 
join b_sql.b_practice.user_info u2 on t.driver_id = u2.user_info_id and  u2.banned = 'No'
where t.request_at  between '2013-10-01' and '2013-10-03'
group by request_at order by request_at)
select request_at, total,canceled,round(canceled*100.0/total,2) as pct from cte


In [0]:
# Import common PySpark SQL functions, types, and Window specification
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Load the trips and user_info tables from Unity Catalog into Spark DataFrames
df_user_info=spark.read.table("b_sql.b_practice.user_info")
df_trips=spark.read.table("b_sql.b_practice.trips")
df_user_info.display()
df_trips.display()

In [0]:
# PySpark Solution: Cancellation rate per day (excluding banned clients/drivers)

# Step 1: Join trips with user_info twice (u1 for client, u2 for driver), both filtered to banned='No'
# Step 2: Group by request_at and aggregate:
#          total     = count(*)
#          cancelled = sum(when status != 'completed' then 1 else 0)
# Step 3: Add cancellation rate = round(cancelled * 100 / total, 2)
df_final=df_trips.alias('t').join(df_user_info.alias("u1"),((col("t.client_id")==col("u1.user_info_id"))&(col("u1.banned")=="No")),"inner").join(df_user_info.alias("u2"),((col("t.driver_id")==col("u2.user_info_id")) & (col("u2.banned")=="No")),"inner")
df_final=(df_final.groupBy("t.request_at")
          .agg(count("*").alias("total"),
            sum(when(col("t.status")!="completed",1).otherwise(0)).alias("cancelled"))
          .withColumn("rate",round(col("cancelled")*100.00/col("total"),2))
          ).display()